# S100 hard-wall x-resolved correlator scaling

This notebook runs the locked $N_x=20$, $N_y=60,50,40$ hard-wall campaign on an A100. At $t=2N_y$ it saves the sample-resolved squared correlator, occupied frame/ranks, and the covariance and occupation spectrum of $A=[0,N_x)\times[0,N_y//2)$. Its v2 resident batches target the A100 aggressively under a strict 30 GiB allocator ceiling, while five-sample durable shards and one rolling five-cycle checkpoint preserve resumability.

## 1. Mount Drive and resolve the locked campaign

In [ ]:
from pathlib import Path
import json
import torch
from google.colab import drive

drive.mount('/content/drive')

BUNDLE = '14_hard_wall_xresolved_correlator_scaling'
REVISION = 'hard_wall_xresolved_nx20_ny40-50-60_a1-1_nsh1_s100_2ny_raster_endpoint_frame_halfcov_occupations_v2_30gib_batched'
BUNDLE_DRIVE_DIR = Path('/content/drive/MyDrive/final_production_new_designs') / BUNDLE
LOCAL_BUNDLE_DIR = Path('/content') / BUNDLE
OUTPUT_ROOT = Path('/content/drive/MyDrive/classA_final_production_outputs') / REVISION
SCRATCH_ROOT = Path('/content/hard_wall_xresolved_correlator_scratch')
REPORT_ONLY = False
MAX_NEW_EXECUTION_BATCHES = None  # e.g. 1 for a controlled short session

CONFIG = {
    'sampling_revision': REVISION,
    'root_seed': 2026091001,
    'Nx': 20,
    'Ny_values': [60, 50, 40],
    'samples_per_Ny': 100,
    'execution_batch_size_by_Ny': {'40': 80, '50': 50, '60': 40},
    'result_shard_size': 5,
    'cycles_rule': '2*Ny',
    'segment_cycles': 5,
    'device': 'cuda:0',
    'dtype': 'complex128',
    'gpu_memory_hard_limit_gib': 30.0,
    'protocol': {
        'DW': True,
        'domain_wall_interval': [5, 15],
        'dw_truncation': True,
        'meas_slab_only': True,
        'nshell': 1,
        'alpha_1': 1.0,
        'alpha_2': 30.0,
        'filling_frac': 0.5,
        'trial_orbitals': 'X',
        'init_mode': 'default',
        'sequence': 'raster_y',
        'perfect_correction': True,
        'postselect': False,
        'postselect_probability': 0.0,
        'n_a': 0.5,
        'state_representation': 'physical_frame',
        'triv_region_local_mode': False,
        'frame_reorthonormalize_interval': 1,
    },
    'observables': {
        'endpoint_only': True,
        'cycles': '[2*Ny]',
        'x_resolved_square_correlator': True,
        'legacy_xavg_square_correlator': True,
        'global_charge': True,
        'half_filling_offset': True,
        'occupied_frame': True,
        'occupied_ranks': True,
        'half_system_covariance': True,
        'half_system_occupation_spectrum': True,
        'half_system_region': '[0,Nx)x[0,Ny//2)',
        'half_system_Ay_rule': 'Ny//2',
        'half_system_covariance_convention': 'C_A=F_A@F_A_dagger',
        'frame_covariance_convention': 'C=F@F_dagger;G=2C-I',
        'full_system_covariance_materialization': False,
        'endpoint_reduced_covariance_materialization': True,
    },
}

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable; select an A100 GPU runtime')
gpu = torch.cuda.get_device_properties(0)
if 'A100' not in gpu.name.upper() or gpu.total_memory < 38 * 1024**3:
    raise RuntimeError(f'Expected an A100 with 40-GB-class memory, found {gpu.name!r} with {gpu.total_memory / 1024**3:.2f} GiB')

print(json.dumps({
    'bundle': BUNDLE,
    'revision': REVISION,
    'bundle_source': str(BUNDLE_DRIVE_DIR),
    'local_stage': str(LOCAL_BUNDLE_DIR),
    'output_root': str(OUTPUT_ROOT),
    'scratch_root': str(SCRATCH_ROOT),
    'gpu': gpu.name,
    'gpu_memory_gib': gpu.total_memory / 1024**3,
    'dtype': CONFIG['dtype'],
    'execution_batches': 7,
    'durable_result_shards': 60,
    'trajectories': 300,
    'estimated_final_occupied_frames_gib': 9.18,
    'estimated_final_half_system_covariances_gib': 4.59,
    'estimated_final_large_payloads_gib': 13.77,
    'required_fresh_drive_free_gib': 20.84,
    'gpu_memory_hard_limit_gib': CONFIG['gpu_memory_hard_limit_gib'],
    'estimated_dynamics_a100_hours_by_Ny': {'40': 1.5, '50': 5.7, '60': 11.5},
    'estimated_total_a100_hours': '21-24',
    'report_only': REPORT_ONLY,
    'max_new_execution_batches': MAX_NEW_EXECUTION_BATCHES,
}, indent=2))

## 2. Stage locally and run or resume

The staged runner executes inside the notebook kernel so `tqdm.auto` uses Colab's native progress display. The outer bar tracks 60 verified five-sample shards; the inner bar tracks the active execution batch and resumes from its last durable five-cycle boundary.

In [ ]:
import importlib.util
import shutil
import sys

if not BUNDLE_DRIVE_DIR.is_dir():
    raise FileNotFoundError(f'Missing Drive bundle: {BUNDLE_DRIVE_DIR}')
if LOCAL_BUNDLE_DIR.exists():
    shutil.rmtree(LOCAL_BUNDLE_DIR)
shutil.copytree(BUNDLE_DRIVE_DIR, LOCAL_BUNDLE_DIR)
config_path = Path('/content/hard_wall_xresolved_correlator_config.json')
config_path.write_text(json.dumps(CONFIG, indent=2, sort_keys=True) + '\n', encoding='utf-8')

runner_args = [
    '--config', str(config_path),
    '--output-root', str(OUTPUT_ROOT),
    '--scratch-root', str(SCRATCH_ROOT),
]
if REPORT_ONLY:
    runner_args.append('--report-only')
if MAX_NEW_EXECUTION_BATCHES is not None:
    runner_args.extend(['--max-new-execution-batches', str(int(MAX_NEW_EXECUTION_BATCHES))])
runner_path = LOCAL_BUNDLE_DIR / 'run_campaign.py'
print('[launch in notebook kernel]', runner_path, ' '.join(runner_args), flush=True)
module_name = '_hard_wall_xresolved_correlator_runner'
sys.modules.pop(module_name, None)
spec = importlib.util.spec_from_file_location(module_name, runner_path)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Could not load staged runner: {runner_path}')
runner = importlib.util.module_from_spec(spec)
sys.modules[module_name] = runner
spec.loader.exec_module(runner)
returncode = runner.main(runner_args)
if returncode:
    raise RuntimeError(f'Runner exited with status {returncode}')
print('[notebook] runner exited successfully', flush=True)

## 3. Release the runtime

In [ ]:
from google.colab import runtime
runtime.unassign()
print('done')
